# Stage 8 — Interpretation & Explainability

This notebook explains the SVM model predictions using:
- **LIME**: Local word-level explanations per class
- **TF-IDF weights**: Global most important words per class
- **Word clouds**: Visual summary of class vocabulary

In [ ]:
# Install libraries
!pip install lime wordcloud --quiet
print('✓ Libraries ready')

In [ ]:
# LIME word-level explanations (one sample per class)
from lime.lime_text import LimeTextExplainer
import numpy as np, matplotlib.pyplot as plt
lime_explainer = LimeTextExplainer(class_names=class_names, random_state=42)
for class_idx in range(num_labels):
    sample_indices = [i for i, y in enumerate(y_test) if y == class_idx]
    if not sample_indices: continue
    exp = lime_explainer.explain_instance(X_test[sample_indices[0]], best_svm.predict_proba, num_features=10, labels=[class_idx])
    fig = exp.as_pyplot_figure(label=class_idx)
    plt.title(f'LIME — Class {le.classes_[class_idx]}')
    plt.tight_layout(); plt.show()

In [ ]:
# TF-IDF feature weights per class
tfidf = best_svm.named_steps['tfidf']
svm_coefs = best_svm.named_steps['clf'].calibrated_classifiers_[0].estimator.coef_
feature_names = np.array(tfidf.get_feature_names_out())
fig, axes = plt.subplots(num_labels, 1, figsize=(10, 5*num_labels))
for i in range(num_labels):
    coef = svm_coefs[i]
    top_pos = np.argsort(coef)[-15:][::-1]
    top_neg = np.argsort(coef)[:15]
    words = np.concatenate([feature_names[top_pos], feature_names[top_neg]])
    weights = np.concatenate([coef[top_pos], coef[top_neg]])
    colors = ['#2ecc71' if w>0 else '#e74c3c' for w in weights]
    axes[i].barh(words, weights, color=colors)
    axes[i].axvline(0, color='black', linewidth=0.8)
    axes[i].set_title(f'Class {le.classes_[i]} — Top words')
plt.tight_layout(); plt.show()

In [ ]:
# Word clouds from SVM weights
from wordcloud import WordCloud
fig, axes = plt.subplots(1, num_labels, figsize=(6*num_labels, 5))
colormaps = ['Blues','Reds','Greens','Purples','Oranges']
for i in range(num_labels):
    coef = svm_coefs[i]
    pos_mask = coef > 0
    word_freq = {w: float(v) for w, v in zip(feature_names[pos_mask], coef[pos_mask])}
    wc = WordCloud(width=500,height=400,background_color='white',colormap=colormaps[i],max_words=60).generate_from_frequencies(word_freq)
    axes[i].imshow(wc, interpolation='bilinear'); axes[i].axis('off')
    axes[i].set_title(f'Class {le.classes_[i]}', fontweight='bold')
plt.suptitle('Key words per class', fontsize=14); plt.tight_layout(); plt.show()